# Neutron Reflectometry — Normalizing Flow Inference

This notebook demonstrates inference with a pre-trained normalizing-flow (NF) model for neutron reflectometry. We use `EasyInferenceModel` to load a trained model and perform posterior sampling on experimental reflectivity data.

Three experimental datasets are tested:
1. **Ni on Si** (`Ni500.dat`) — 1-layer structure, dq/q = 10%
2. **Ni on Glass** (`Ni_on_glass.dat`) — 1-layer structure, dq/q = 1.5%
3. **Pointwise resolution data** (`s000004_experimental_curve.dat`) — with per-point dR and dQ

> **Note:** This notebook should be run from the repository root directory.

## 1. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from reflectorch import EasyInferenceModel, interp_reflectivity

torch.manual_seed(0)

## 2. Load the Inference Model

We initialize an `EasyInferenceModel` for a model trained on neutron reflectometry with a mean-conditioned scaler and a standard 1-layer parameterization.

In [ ]:
config_name = 'nf_config_mixed_mean_conditioned.yaml'

inference_model = EasyInferenceModel(
    config_name=config_name,
    device='cpu',
    weights_format='safetensors',
)

## 3. Define Visualization Helpers

Two reusable functions for visualizing inference results:
- `postprocess` — shows all sampled curves/SLD profiles with parameter statistics
- `postprocess_extremes` — highlights the best and worst samples (by log-likelihood)

In [ ]:
def postprocess(prediction_dict, q_exp, curve_exp, sigmas_exp=None):
    """Plot sampled reflectivity curves and SLD profiles with parameter statistics."""
    sampled_curves = prediction_dict['sampled_curves']
    sampled_slds = prediction_dict['sampled_sld_profiles']
    sld_xaxis = prediction_dict['sampled_sld_xaxis']
    q_plot = prediction_dict['q_plot_pred']
    pred_params = prediction_dict['predicted_params_array']
    param_names = prediction_dict['param_names']

    # Parameter statistics
    mean_params = np.mean(pred_params, axis=0)
    std_params = np.std(pred_params, axis=0)

    print(f"{'Parameter':<20} | {'Mean':<12} | {'Std Dev':<12}")
    print('-' * 50)
    for name, mean, std in zip(param_names, mean_params, std_params):
        print(f"{name:<20} | {mean:<12.4f} | {std:<12.4f}")

    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # --- Reflectivity ---
    ax = axes[0]
    ax.set_yscale('log')
    ax.set_xlabel('q [$\\AA^{-1}$]', fontsize=14)
    ax.set_ylabel('R(q)', fontsize=14)
    ax.set_title('Reflectivity', fontsize=16)

    yerr = sigmas_exp if sigmas_exp is not None else None
    ax.errorbar(q_exp, curve_exp, yerr=yerr, fmt='o', color='black',
                markersize=3, alpha=0.6, label='Exp. Data', zorder=10)

    for i, curve in enumerate(sampled_curves):
        ax.plot(q_plot, curve, color='red', alpha=0.15, linewidth=0.8,
                label='Predicted Samples' if i == 0 else None)

    ax.legend(fontsize=12)
    ax.grid(alpha=0.2, which='both')

    # --- SLD Profile ---
    ax = axes[1]
    ax.set_xlabel('z [$\\AA$]', fontsize=14)
    ax.set_ylabel('SLD [$10^{-6} \\AA^{-2}$]', fontsize=14)
    ax.set_title('SLD Profile', fontsize=16)

    for i, sld in enumerate(sampled_slds):
        ax.plot(sld_xaxis, sld, color='red', alpha=0.15, linewidth=0.8,
                label='Predicted Samples' if i == 0 else None)

    ax.legend(fontsize=12)
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

In [ ]:
def postprocess_extremes(prediction_dict, q_exp, curve_exp):
    """Compare best and worst samples by log-likelihood."""
    log_likelihoods = prediction_dict.get('log_likelihoods')
    pred_params = prediction_dict['predicted_params_array']
    param_names = prediction_dict['param_names']
    sampled_curves = prediction_dict['sampled_curves']
    sampled_slds = prediction_dict['sampled_sld_profiles']
    q_plot = prediction_dict['q_plot_pred']
    sld_xaxis = prediction_dict['sampled_sld_xaxis']

    if log_likelihoods is not None:
        best_idx = np.argmax(log_likelihoods)
        worst_idx = np.argmin(log_likelihoods)
        print(f'Best Sample:  idx={best_idx}  (LogL: {log_likelihoods[best_idx]:.4f})')
        print(f'Worst Sample: idx={worst_idx}  (LogL: {log_likelihoods[worst_idx]:.4f})')
    else:
        print('Warning: log_likelihoods not found. Using first and last samples.')
        best_idx, worst_idx = 0, -1

    # Parameter comparison
    print(f"\n{'Parameter':<20} | {'Best':<12} | {'Worst':<12}")
    print('-' * 50)
    for i, name in enumerate(param_names):
        print(f"{name:<20} | {pred_params[best_idx, i]:<12.4f} | {pred_params[worst_idx, i]:<12.4f}")

    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Reflectivity ---
    ax = axes[0]
    ax.set_yscale('log')
    ax.set_xlabel('q [$\\AA^{-1}$]', fontsize=14)
    ax.set_ylabel('R(q)', fontsize=14)
    ax.set_title('Reflectivity', fontsize=16)

    ax.errorbar(q_exp, curve_exp, fmt='o', color='black',
                markersize=4, alpha=0.6, label='Exp. Data', zorder=10)
    ax.plot(q_plot, sampled_curves[best_idx], color='green', linewidth=2, label='Best Sample')
    ax.plot(q_plot, sampled_curves[worst_idx], color='red', linestyle='--', linewidth=2, label='Worst Sample')

    ax.legend(fontsize=12)
    ax.grid(alpha=0.2, which='both')

    # --- SLD Profile ---
    ax = axes[1]
    ax.set_xlabel('z [$\\AA$]', fontsize=14)
    ax.set_ylabel('SLD [$10^{-6} \\AA^{-2}$]', fontsize=14)
    ax.set_title('SLD Profile', fontsize=16)

    ax.plot(sld_xaxis, sampled_slds[best_idx], color='green', linewidth=2, label='Best Sample')
    ax.plot(sld_xaxis, sampled_slds[worst_idx], color='red', linestyle='--', linewidth=2, label='Worst Sample')

    ax.legend(fontsize=12)
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

## 4. Example 1 — Ni on Si (Ni500.dat)

A 1-layer structure of Ni on Si (air as ambient), measured at dq/q = 0.1 (10%).

### Load and plot experimental data

In [ ]:
data = np.loadtxt('exp_data/Ni500.dat', delimiter='\t', skiprows=0)

q_exp = data[..., 0]
curve_exp = data[..., 1]
sigmas_exp = data[..., 2]

print(f'Shape: {data.shape}  |  q range: [{q_exp.min():.4f}, {q_exp.max():.4f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.set_yscale('log')
ax.set_xlabel('q [$\\AA^{-1}$]', fontsize=14)
ax.set_ylabel('R(q)', fontsize=14)

ax.errorbar(q_exp, curve_exp, yerr=sigmas_exp, fmt='o', markersize=2, linewidth=1,
            color='blue', ecolor='green', elinewidth=1, capsize=2, label='Experimental data')

ax.legend(fontsize=12)
ax.grid(alpha=0.2, which='both')
plt.tight_layout()
plt.show()

### Set prior bounds, interpolate, and run inference

In [ ]:
prior_bounds = [
    (300., 900.),          # layer thickness
    (0., 20.), (0., 20.),  # interlayer roughnesses (top to bottom)
    (9., 11.), (3., 5.),   # layer SLDs: Ni, glass
    (0.9, 1.1),            # intensity scaling factor
    (-10.0, -4.0),         # log10 background
]

q_model, exp_curve_interp = inference_model.interpolate_data_to_model_q(q_exp, curve_exp)
print(f'Model q grid: {q_model.shape}  |  range: [{q_model.min():.4f}, {q_model.max():.4f}]')

In [ ]:
prediction_dict = inference_model.preprocess_and_sample(
    reflectivity_curve=exp_curve_interp,
    q_values=q_model,
    num_samples=1000,
    prior_bounds=prior_bounds,
    q_resolution=0.1,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
)

### Results

In [ ]:
postprocess(prediction_dict, q_exp, curve_exp, sigmas_exp=sigmas_exp)

In [ ]:
# Reference results (from previous reflectorch runs):
#
# Thickness L1   -> Predicted: 498.53   Polished: 497.37
# Roughness L1   -> Predicted: 6.45     Polished: 5.65
# Roughness sub  -> Predicted: 7.68     Polished: 9.39
# SLD L1         -> Predicted: 10.24    Polished: 10.24
# SLD sub        -> Predicted: 3.92     Polished: 3.87

## 5. Example 2 — Ni on Glass

### Load experimental data

In [ ]:
data = np.loadtxt('exp_data/Ni_on_glass.dat', delimiter='\t', skiprows=0)

q_exp = data[..., 0]
curve_exp = data[..., 1]

print(f'Shape: {data.shape}  |  q range: [{q_exp.min():.4f}, {q_exp.max():.4f}]')

### Set prior bounds, interpolate, and run inference

In [ ]:
prior_bounds = [
    (300., 900.),          # layer thickness
    (0., 20.), (0., 20.),  # interlayer roughnesses (top to bottom)
    (9., 11.), (3., 5.),   # layer SLDs: Ni, glass
    (0.9, 1.1),            # intensity scaling factor
    (-10, -4),             # log10 background
]

q_model, exp_curve_interp = inference_model.interpolate_data_to_model_q(q_exp, curve_exp)
print(f'Model q grid: {q_model.shape}  |  range: [{q_model.min():.4f}, {q_model.max():.4f}]')

In [ ]:
prediction_dict = inference_model.preprocess_and_sample(
    reflectivity_curve=exp_curve_interp,
    q_values=q_model,
    num_samples=1000,
    prior_bounds=prior_bounds,
    q_resolution=0.015,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
)

### Results

In [ ]:
postprocess(prediction_dict, q_exp, curve_exp)

In [ ]:
# Reference results (from previous reflectorch runs):
#
# Thickness L1     -> Predicted: 812.66   Polished: 826.98
# Roughness L1     -> Predicted: 8.42     Polished: 8.14
# Roughness sub    -> Predicted: 1.35     Polished: 0.00
# SLD L1           -> Predicted: 9.05     Polished: 9.17
# SLD sub          -> Predicted: 3.85     Polished: 4.00
# r_scale          -> Predicted: 0.97     Polished: 0.91
# log10_background -> Predicted: -4.17    Polished: -4.00

### Importance sampling

Re-run with 10,000 samples, log-likelihood computation, and importance sampling to identify the best and worst posterior samples.

In [ ]:
prediction_dict = inference_model.preprocess_and_sample(
    reflectivity_curve=exp_curve_interp,
    q_values=q_model,
    num_samples=10000,
    prior_bounds=prior_bounds,
    q_resolution=0.015,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
    calc_log_likelihoods=True,
    enable_importance_sampling=True,
)

In [ ]:
postprocess_extremes(prediction_dict, q_exp, curve_exp)

## 6. Example 3 — Pointwise Resolution Data

This dataset includes per-point measurement uncertainties (dR) and resolution values (dQ), enabling pointwise resolution smearing during inference.

### Load experimental data

In [ ]:
data = np.loadtxt('exp_data/s000004_experimental_curve.dat', skiprows=1)

q_exp = data[..., 0]
curve_exp = data[..., 1]
sigmas_exp = data[..., 2]
q_res_exp = data[..., 3]

print(f'Shape: {data.shape}  |  q range: [{q_exp.min():.4f}, {q_exp.max():.4f}]')
print(f'dR/R range: [{(sigmas_exp / curve_exp).min():.4f}, {(sigmas_exp / curve_exp).max():.4f}]')
print(f'dq/q range: [{(q_res_exp / q_exp).min():.4f}, {(q_res_exp / q_exp).max():.4f}]')

### Set prior bounds, interpolate, and run inference

Known ground truth from Refl1D fit:

| Layer | SLD (Å⁻²) | Thickness (Å) | Roughness (Å) |
|-------|-----------|---------------|----------------|
| fronting | 3.50e-06 | ∞ | 8.04 |
| layer1 | 3.57e-06 | 28.63 | 7.04 |
| backing | 9.80e-06 | ∞ | — |

In [ ]:
prior_bounds = [
    (10., 50.),            # layer thickness
    (0., 15.), (0., 15.),  # interlayer roughnesses (top to bottom)
    (3.0, 4.0),            # layer SLD: L1
    (9.0, 11.0),           # layer SLD: backing
    (0.9, 1.1),            # intensity scaling factor
    (-10, -4),             # log10 background
]

In [ ]:
# Interpolate both curve and sigmas to the model Q grid
q_model, exp_curve_interp, sigmas_interp = inference_model.interpolate_data_to_model_q(
    q_exp, curve_exp, sigmas_exp=sigmas_exp,
)

# Interpolate pointwise resolution
q_res_interp = np.interp(q_model, q_exp, q_res_exp)

print(f'Model q grid: {q_model.shape}  |  range: [{q_model.min():.4f}, {q_model.max():.4f}]')
print(f'dq/q range (interpolated): [{(q_res_interp / q_model).min():.4f}, {(q_res_interp / q_model).max():.4f}]')

In [ ]:
prediction_dict = inference_model.preprocess_and_sample(
    reflectivity_curve=exp_curve_interp,
    q_values=q_model,
    sigmas=sigmas_interp,
    q_resolution=q_res_interp,
    num_samples=1000,
    prior_bounds=prior_bounds,
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
    calc_log_likelihoods=True,
    enable_importance_sampling=True,
)

### Results

In [ ]:
postprocess_extremes(prediction_dict, q_exp, curve_exp)

In [ ]:
print(f"Log evidence: {prediction_dict['log_evidence']:.4f}")